In [1]:
from abc import ABC, abstractmethod
from typing import List, Any, Dict
from pathlib import Path
from langchain_core.documents import Document
from langchain_community.document_loaders import (
    TextLoader,
    PyMuPDFLoader,
    UnstructuredMarkdownLoader,
    UnstructuredWordDocumentLoader,
    BSHTMLLoader)

/var/folders/xf/_6m_tb052fn5p84tlt773rqr0000gn/T/ipykernel_4009/2221561814.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [ ]:
class BaseLoader:
    def __init__(self, config: Dict[str, Any] | None = None):
        self.config = config or {}


    def normalize_metadata(self, source: str, extra: Dict[str, Any] | None = None) -> Dict[str,Any]:
        meta = {
            "source": source,
            "ingested_at": None
        }
        if extra:
            meta.update(extra)
        return meta


In [ ]:
def normalize_docs(docs: List[Document], source: str, source_type: str) -> List[Document]:
    normalized = []
    for i, doc in enumerate(docs):
        page = doc.metadata.get("page", None)
        title = doc.metadata.get("title", None)
        file_name = doc.metadata.get("source", source)

        meta = {
            "source": source,
            "source_type": source_type,
            "file_name": file_name,
            "page": page,
            "title": title,
            "doc_index": i
        }
        meta.update(doc.metadata)
        normalized.append(
            Document(
                page_content=doc.page_content.strip(),
                metadata = meta
                ))

    return normalized



In [5]:
from pathlib import Path
import os

In [6]:
root = Path().parent.resolve().as_posix()

print(root)

os.chdir("/Users/harendrakumar/Documents/rag_framework/")

/Users/harendrakumar/Documents/rag_framework/notebooks


In [7]:
os.curdir

'.'

In [ ]:
class FileLoader(BaseLoader):
    def load(self, source: str) -> List[Document]:
        path = Path(source)

        if not path.exists():
            raise ValueError(f"Soure file not found: {source}")

        suffix = path.suffix.lower()

        if suffix == ".txt":
            loader=TextLoader(str(path), encoding="utf-8")
            docs = loader.load()

        elif suffix == ".md":
            loader = UnstructuredMarkdownLoader(str(path), mode="elements")
            docs = loader.load()

        elif suffix == '.docx':
            loader = UnstructuredWordDocumentLoader(str(path))
            docs = loader.load()

        elif suffix == '.pdf':
            loader = PyMuPDFLoader(str(path))
            docs = loader.load()

        elif suffix in ('.html', '.htm'):
            loader = BSHTMLLoader(str(path), open_encoding="utf-8")
            docs = loader.load()

        else:
            raise ValueError(f"Unsupported file type: {suffix}")

        return normalize_docs(docs=docs, source=source, source_type='file')

In [ ]:
class FolderLoader(BaseLoader):
    def __init__(self, allowed_extensions = None):
        self.allowed_extensions = allowed_extensions or {'.txt', '.md', '.pdf', '.docx', '.html', '.htm'}
        self.file_loader = FileLoader()


    def load(self, folder_path: str) -> List[Document]:
        folder = Path(folder_path)

        if not folder.exists():
            raise FileNotFoundError(f"Folder not found: {folder_path}")

        docs: List[Document] = []
        for file_path in sorted(folder.rglob("*")):
            if file_path.is_file() and file_path.suffix.lower() in self.allowed_extensions:
                docs.extend(self.file_loader.load(str(file_path)))

        return docs


In [2]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
class WebLoader(BaseLoader):
    def load(self, source: str) -> List[Document]:
        loader = WebBaseLoader(source)
        docs = loader.load()
        return normalize_docs(docs, source, "web")

NameError: name 'BaseLoader' is not defined

In [ ]:
from src.loaders.registry import LoaderRegistry

In [ ]:
loader = LoaderRegistry()

documents =loader.load("/Users/harendrakumar/Documents/rag_framework/data/RAG_101_Chunking_Strategies.pdf")

In [ ]:
len(documents)

In [ ]:
# print(len(documents))
# for doc in documents[:2]:
#     print(doc.metadata)
#     print(doc.page_content[:300])

In [ ]:
# for i, doc in enumerate(docs):
#     source = doc.metadata.get("source", None)
#     source_type = doc.metadata.get("source_type", None)
#     file_name = doc.metadata.get("file_name", None)
#     page = doc.metadata.get("page", None)
#     title = doc.metadata.get("title", None)

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

In [ ]:
chunk_size=800
chunk_overlap=120

splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size = chunk_size,
    chunk_overlap = chunk_overlap,
    length_function = len,
    is_separator_regex=False
)

In [ ]:
chunks = []

for doc_index, doc in enumerate(documents):
    split_docs = splitter.split_documents([doc])
    for chunk_index, chunk in enumerate(split_docs):
        meta = dict(chunk.metadata or {})
        meta.update({
            "doc_index": doc_index,
            "chunk_index": chunk_index,
            "chunk_size": chunk_size,
            "chunk_overlap": chunk_overlap,
            "strategy": "fixed_size"
        })
        chunks.append(Document(page_content=chunk.page_content.strip(), metadata=meta))

In [ ]:
chunks[1]

In [ ]:
from langchain_core.documents import Document
from src.chunking.dynamic_chunker import DynamicChunker

documents = [
    Document(
        page_content="""
        # AI and RAG systems

        Retrieval-augmented generation is a practical architecture for grounding LLMs.
        It combines retrieval from documents with generation and prompt assembly.
        This is extremely useful when you need domain-specific and up-to-date information.
        """,
        metadata={"source": "doc1.txt", "source_type": "text"}
    ),
    Document(
        page_content="""This is another document about chunking strategies in vector search. We can discuss semantic chunking, recursive chunking, embeddings, and retrieval-quality tuning.""",
        metadata={"source": "doc2.txt", "source_type": "text"}
    )
]

chunker = DynamicChunker()
chunks = chunker.split(documents)

print("Total chunks:", len(chunks))
for chunk in chunks[:3]:
    print("----")
    print(chunk.metadata)
    print(chunk.page_content[:200])

In [ ]:
chunker = DynamicChunker()
chunks = chunker.split(documents)

print("Total chunks:", len(chunks))
for chunk in chunks[:3]:
    print("----")
    print(chunk.metadata)
    print(chunk.page_content[:200])

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
kwargs = {
    "OPENAI_API_KEY": os.getenv("OPENAI_API_KEY"),
    "BASE_URL": os.getenv("BASE_URL"),
    "EMBEDDING_MODEL": os.getenv("EMBEDDING_MODEL")
    }

In [8]:
from langchain_core.documents import Document
# from src.embeddings.huggingface_embedder import HuggingFaceEmbedder
from src.embeddings.openai_embedder import OpenAIEmbedder
from src.vectorstore.faiss_store import FAISSVectorStore
from src.retrieval.semantic_retriever import SemanticRetriever

docs = [
    Document(
        page_content="RAG systems combine retrieval and generation to ground LLM responses in external knowledge.",
        metadata={"source": "doc1.txt", "source_type": "file", "category": "rag"}
    ),
    Document(
        page_content="Chunking is important for controlling context windows and improving retrieval quality.",
        metadata={"source": "doc2.txt", "source_type": "file", "category": "chunking"}
    ),
    Document(
        page_content="FAISS is a fast vector database for embeddings-based similarity search.",
        metadata={"source": "doc3.txt", "source_type": "file", "category": "vector_db"}
    )
]

# embedder = HuggingFaceEmbedder()
embedder = OpenAIEmbedder()
vectorstore = FAISSVectorStore(embedding_model=embedder)

vectorstore.add_documents(docs)

retriever = SemanticRetriever(vectorstore)
results = retriever.retrieve("How does retrieval work in RAG?", k=2)

for result in results:
    print("METADATA:", result.metadata)
    print("CONTENT:", result.page_content[:300])
    print("----")

METADATA: {'source': 'doc1.txt', 'source_type': 'file', 'category': 'rag'}
CONTENT: RAG systems combine retrieval and generation to ground LLM responses in external knowledge.
----
METADATA: {'source': 'doc2.txt', 'source_type': 'file', 'category': 'chunking'}
CONTENT: Chunking is important for controlling context windows and improving retrieval quality.
----
